In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.appName("Skripsi").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/28 10:54:39 WARN Utils: Your hostname, pc, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/28 10:54:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 10:54:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)

In [4]:
valid_temps = [-20, -10, 0, 10, 25, 40]

closest_val = F.lit(valid_temps[0])
min_diff = F.abs(F.col("Ambient_Temp_degC") - F.lit(valid_temps[0]))

for t in valid_temps[1:]:
    current_diff = F.abs(F.col("Ambient_Temp_degC") - F.lit(t))
    
    closest_val = F.when(current_diff < min_diff, F.lit(t)).otherwise(closest_val)
    min_diff = F.when(current_diff < min_diff, current_diff).otherwise(min_diff)

df = df.withColumn("Ambient_Temp_degC", closest_val)

In [7]:
import os
from pyspark.sql import Window
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns

# Buat folder 'figure' jika belum ada
output_folder = "figure"
os.makedirs(output_folder, exist_ok=True)

window_spec = Window.partitionBy("Test_Cell", "Ambient_Temp_degC").orderBy("Time_s")

df_with_changes = df.withColumn(
    "is_new_episode",
    F.when(F.col("Cycle_Label") != F.lag("Cycle_Label", 1).over(window_spec), 1).otherwise(0)
)

df_clean = df_with_changes.withColumn(
    "Episode_ID",
    F.sum("is_new_episode").over(window_spec)
)

combinations = df_clean.select("Test_Cell", "Ambient_Temp_degC", "Cycle_Label").distinct().collect()

for row in combinations:
    cell = row["Test_Cell"]
    temp = row["Ambient_Temp_degC"]
    cycle = row["Cycle_Label"]
    
    if not cycle:
        continue
        
    print(f"Generating plot for {cell} | {temp}°C | Cycle: {cycle}...")
    
    filtered_df = df_clean.filter(
        (df_clean["Test_Cell"] == cell) & 
        (df_clean["Ambient_Temp_degC"] == temp) &
        (df_clean["Cycle_Label"] == cycle)
    )
    
    pdf = filtered_df.select("Time_s", "SOC", "Episode_ID").toPandas()
    pdf = pdf.sort_values(by="Time_s")
    
    if pdf.empty:
        continue
        
    plt.figure(figsize=(12, 6))
    
    sns.lineplot(
        data=pdf, 
        x="Time_s", 
        y="SOC", 
        hue="Episode_ID", 
        units="Episode_ID", 
        estimator=None, 
        linewidth=1.5,
        palette="tab10"
    )
    
    plt.title(f"SOC Profile - Cell: {cell} | Temp: {temp}°C | Cycle: {cycle}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("State of Charge (SOC)")
    plt.legend(title="Episode ID", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    
    # KUNCI PERUBAHAN: Menggabungkan nama folder dengan nama file
    file_path = os.path.join(output_folder, f"fig_{cell}_{temp}_{cycle}.png")
    plt.savefig(file_path)
    plt.close() # Menggunakan plt.close() menggantikan plt.show() agar hemat RAM di Colab

Generating plot for m1000 | 25°C | Cycle: CC_CV_charge...


Generating plot for m1000 | 25°C | Cycle: Other...


Generating plot for m1000 | 40°C | Cycle: CC_CV_charge...


Generating plot for m1000 | -10°C | Cycle: C_20_Discharge_Charge...


Generating plot for m1000 | -10°C | Cycle: HPPC...


...

Generating plot for m80 | 40°C | Cycle: HWGRADE1...


Generating plot for m80 | 40°C | Cycle: REORDERED4...


Generating plot for m80 | 40°C | Cycle: REORDERED8...


Generating plot for m80 | 40°C | Cycle: REORDERED3...


Generating plot for m80 | 40°C | Cycle: 0.5C_Discharge...


In [8]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan folder penyimpanan sudah siap
output_folder = "figure"
os.makedirs(output_folder, exist_ok=True)

# Definisikan daftar kolom berdasarkan jenis visualisasinya
numeric_cols = ["Voltage_V", "Current_A", "Ah", "SOC", "Power_W", "Wh", "Battery_Temp_degC"]
categorical_cols = ["Cycle_Label", "Test_Cell", "Ambient_Temp_degC"]

# --- PART 1: HISTOGRAM (KOLOM NUMERIK) ---
# Mengambil sampel data (misal 5%) agar proses toPandas() ringan dan aman di RAM
sampled_df = df.select(numeric_cols).sample(withReplacement=False, fraction=0.05, seed=42).toPandas()

for col in numeric_cols:
    print(f"Generating histogram for {col}...")
    
    plt.figure(figsize=(10, 5))
    sns.histplot(data=sampled_df, x=col, kde=True, color="skyblue", bins=50)
    
    plt.title(f"Distribution of {col} (Sampled Data)")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, f"hist_{col}.png"))
    plt.close()


# --- PART 2: BAR CHART (KOLOM KATEGORIKAL) ---
# Untuk data kategorikal, kita hitung frekuensinya langsung di PySpark (Sangat hemat RAM)
for col in categorical_cols:
    print(f"Generating bar chart for {col}...")
    
    # Hitung jumlah kemunculan setiap kategori menggunakan PySpark
    counts_df = df.groupBy(col).count().orderBy(F.desc("count")).toPandas()
    
    plt.figure(figsize=(10, 5))
    
    # Gunakan barplot berdasarkan hasil agregasi ringkas tadi
    sns.barplot(data=counts_df, x=col, y="count", palette="viridis")
    
    plt.title(f"Frequency of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45 if col == "Cycle_Label" else 0) # Putar teks jika label panjang
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, f"bar_{col}.png"))
    plt.close()

print("All histograms and bar charts have been saved to the 'figure' folder!")

Generating histogram for Voltage_V...
Generating histogram for Current_A...
Generating histogram for Ah...
Generating histogram for SOC...
Generating histogram for Power_W...
Generating histogram for Wh...
Generating histogram for Battery_Temp_degC...
Generating bar chart for Cycle_Label...


/tmp/ipykernel_11325/2056565445.py:44: FutureWarning:                           

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=counts_df, x=col, y="count", palette="viridis")


Generating bar chart for Test_Cell...


/tmp/ipykernel_11325/2056565445.py:44: FutureWarning:                           

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=counts_df, x=col, y="count", palette="viridis")


Generating bar chart for Ambient_Temp_degC...


/tmp/ipykernel_11325/2056565445.py:44: FutureWarning:                           

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=counts_df, x=col, y="count", palette="viridis")


All histograms and bar charts have been saved to the 'figure' folder!


In [9]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan folder penyimpanan sudah siap
output_folder = "figure"
os.makedirs(output_folder, exist_ok=True)

# 1. Ambil sampel data (misal 2% - 5%) agar plotting ringan dan tidak merusak RAM
# Sesuaikan fraction (0.02 = 2%) tergantung seberapa besar total baris data kamu
sampled_df = df.select(
    "Voltage_V", "Current_A", "Power_W", "Battery_Temp_degC", "SOC", "Cycle_Label"
).sample(withReplacement=False, fraction=0.02, seed=42).toPandas()


# --- SCATTER PLOT 1: Voltage vs Current (Karakteristik V-I Baterai) ---
print("Generating scatter plot for Voltage vs Current...")
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sampled_df, 
    x="Current_A", 
    y="Voltage_V", 
    hue="Cycle_Label", # Membedakan warna berdasarkan siklus kerja
    alpha=0.5,         # Transparansi agar titik yang bertumpuk terlihat jelas
    edgecolor=None     # Menghilangkan garis tepi titik agar plot lebih halus
)
plt.title("Scatter Plot: Voltage vs Current (Sampled Data)")
plt.xlabel("Current (A)")
plt.ylabel("Voltage (V)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(title="Cycle Label", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "scatter_voltage_current.png"))
plt.close()


# --- SCATTER PLOT 2: Battery Temp vs Power (Efek Termal dari Daya) ---
print("Generating scatter plot for Battery Temp vs Power...")
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sampled_df, 
    x="Power_W", 
    y="Battery_Temp_degC", 
    hue="Cycle_Label", 
    alpha=0.5, 
    edgecolor=None
)
plt.title("Scatter Plot: Battery Temperature vs Power (Sampled Data)")
plt.xlabel("Power (W)")
plt.ylabel("Battery Temperature (°C)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(title="Cycle Label", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "scatter_temp_power.png"))
plt.close()


# --- SCATTER PLOT 3: SOC vs Voltage (Kurva OCV / Tegangan terhadap Kapasitas) ---
print("Generating scatter plot for SOC vs Voltage...")
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sampled_df, 
    x="SOC", 
    y="Voltage_V", 
    hue="Cycle_Label", 
    alpha=0.5, 
    edgecolor=None
)
plt.title("Scatter Plot: Voltage vs SOC (Sampled Data)")
plt.xlabel("State of Charge (SOC)")
plt.ylabel("Voltage (V)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(title="Cycle Label", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_folder, "scatter_voltage_soc.png"))
plt.close()

print("All scatter plots have been saved successfully to the 'figure' folder!")

Generating scatter plot for Voltage vs Current...
Generating scatter plot for Battery Temp vs Power...
Generating scatter plot for SOC vs Voltage...
All scatter plots have been saved successfully to the 'figure' folder!


In [10]:
from pyspark.sql import functions as F

# 1. Ambil kunci episode unik termasuk Episode_ID
episode_keys = df_clean.select('Test_Cell', 'Cycle_Label', 'Ambient_Temp_degC', 'Episode_ID').distinct()

# 2. Bagi episode secara acak (80% train, 20% test)
train_keys, test_keys = episode_keys.randomSplit([0.8, 0.2], seed=42)

# 3. Lakukan Join menggunakan keempat kolom kunci tersebut
df_train = df_clean.join(train_keys, on=['Test_Cell', 'Cycle_Label', 'Ambient_Temp_degC', 'Episode_ID'], how='inner')
df_test = df_clean.join(test_keys, on=['Test_Cell', 'Cycle_Label', 'Ambient_Temp_degC', 'Episode_ID'], how='inner')

# 4. Hitung jumlah baris data hasil pemisahan
print(f"Train rows : {df_train.count()}")
print(f"Test rows  : {df_test.count()}")

26/05/28 13:18:11 WARN RowBasedKeyValueBatch: Failed to allocate page (2097136 bytes).
26/05/28 13:18:12 ERROR Executor: Exception in task 1.0 in stage 1020.0 (TID 8815)
org.apache.spark.memory.SparkOutOfMemoryError: [UNABLE_TO_ACQUIRE_MEMORY] Unable to acquire 262144 bytes of memory, got 89227. SQLSTATE: 53200
	at org.apache.spark.errors.SparkCoreErrors$.outOfMemoryError(SparkCoreErrors.scala:456)
	at org.apache.spark.errors.SparkCoreErrors.outOfMemoryError(SparkCoreErrors.scala)
	at org.apache.spark.memory.MemoryConsumer.throwOom(MemoryConsumer.java:157)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:98)
	at org.apache.spark.unsafe.map.BytesToBytesMap.allocate(BytesToBytesMap.java:868)
	at org.apache.spark.unsafe.map.BytesToBytesMap.<init>(BytesToBytesMap.java:202)
	at org.apache.spark.unsafe.map.BytesToBytesMap.<init>(BytesToBytesMap.java:209)
	at org.apache.spark.sql.execution.UnsafeFixedWidthAggregationMap.<init>(UnsafeFixedWidthAggregationMap.java:96

Py4JJavaError: An error occurred while calling o8184.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 1020.0 failed 1 times, most recent failure: Lost task 1.0 in stage 1020.0 (TID 8815) (10.255.255.254 executor driver): org.apache.spark.memory.SparkOutOfMemoryError: [UNABLE_TO_ACQUIRE_MEMORY] Unable to acquire 262144 bytes of memory, got 89227. SQLSTATE: 53200
	at org.apache.spark.errors.SparkCoreErrors$.outOfMemoryError(SparkCoreErrors.scala:456)
	at org.apache.spark.errors.SparkCoreErrors.outOfMemoryError(SparkCoreErrors.scala)
	at org.apache.spark.memory.MemoryConsumer.throwOom(MemoryConsumer.java:157)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:98)
	at org.apache.spark.unsafe.map.BytesToBytesMap.allocate(BytesToBytesMap.java:868)
	at org.apache.spark.unsafe.map.BytesToBytesMap.<init>(BytesToBytesMap.java:202)
	at org.apache.spark.unsafe.map.BytesToBytesMap.<init>(BytesToBytesMap.java:209)
	at org.apache.sp

In [ ]:
df.show(5)

+--------------------+------+----------------+-----------------+------+-----------------+-----------------+------+-----------------+-----------------+------------+---------+
|           TimeStamp|Time_s|       Voltage_V|        Current_A|    Ah|              SOC|          Power_W|    Wh|Ambient_Temp_degC|Battery_Temp_degC| Cycle_Label|Test_Cell|
+--------------------+------+----------------+-----------------+------+-----------------+-----------------+------+-----------------+-----------------+------------+---------+
|10/29/2021 16:25:...|     0|2.78867142857143|          -2.0959|3.7508|0.123643079252387|-5.84305819571432|12.825|             25.0|  5.6138285714286|CC_CV_charge|    m1000|
|10/29/2021 16:25:...|     1|2.80099957862128|-2.24341190186706|3.7508|0.123643079252387|-6.28110464284739|12.825|             25.0| 5.59957608967206|CC_CV_charge|    m1000|
|10/29/2021 16:25:...|     2|2.80642753618361|-2.20590919538316|3.7508|0.123643079252387|-6.17610456523376|12.825|             25.